# core

> Claude code api backend for fastllm

FastLLM's `claude_code` provider, as a thin adapter over `fastclaude`: `claude_mk_payload` hands the canonical history and the chat's tool callables to `astream`, and `claude_acollect_stream` adapts the run's raw events to FastLLM's delta stream. Claude Code owns the tool loop: callable tools execute in-process mid-run and stream real results, every tool call is marked server-executed so FastLLM starts no second loop, and `ClaudeCodeCallback` reshapes each turn's history into the assistant/tool message shape the rest of the stack already understands. A request must end with a real user prompt; the old deferred-tool continuation is gone.

In [ ]:
#| default_exp core

In [ ]:
#| export
import asyncio
from fastcore.utils import *
from fastllm.types import *
from aidialog.msg_parts import Msg, ToolUse, ToolResult, Completion
from fastllm.anthropic import norm_sse_event, norm_tool_calls, norm_parts, norm_finish, norm_usage, finalize_usage, delta_index_fn, cost
from fastllm.streaming import mk_acollect_stream, Delta
from fastllm.chat import ChatCallback, defaults
from fastspec.errors import APIError
from fastclaude.core import astream, MCP_PREFIX, SERVER_TOOLS

In [ ]:
from fastllm.chat import acomplete, lite_mk_func, AsyncChat, mk_msgs, defaults
from aidialog.msg_parts import FullResponse, hist2fmt
from fastcore.test import *
from aidialog.msg_parts import Text, Thinking

`claude_mk_payload` passes the canonical history straight through: `astream` compiles, files, and resumes it itself. FastLLM supplies tool schemas; the matching callables come from the chat's namespace, injected as `ns` by `ClaudeCodeCallback` below, and each pair becomes a fastclaude `(schema, callable)` tool. The namespace may be dict-shaped without being a dict (ipyai's `_BridgeNS` proxy is empty as a dict but resolves every name), so only `None` means absent. `web_search_options` enables Claude Code's own search tools:

In [ ]:
#| export
def claude_mk_payload(msgs, model, stream=False, **kwargs):
    "Build `astream` inputs: canonical history, paired tool schemas and callables, native search tools"
    ns = ifnone(kwargs.get('ns'), {})
    tools = [(dict(name=f['name'], description=f.get('description',''), inputSchema=f.get('parameters', {})), fn)
        for t in (kwargs.get('tools') or []) if (f := t.get('function')) and (fn := ns.get(f['name']))]
    native = SERVER_TOOLS if kwargs.get('web_search_options') is not None else ()
    return dict(msgs=list(msgs), model=model, system=kwargs.get('system') or '', tools=tools or None, native_tools=native)

In [ ]:
def simple_add(a: int, b: int) -> int:
    "Add two numbers"
    return a + b

p = claude_mk_payload(mk_msgs(['What is 2+2?']), 'claude-sonnet-4-6',
    tools=[lite_mk_func(simple_add)], ns=dict(simple_add=simple_add))
test_eq(p['tools'][0][0]['name'], 'simple_add')
test_is(p['tools'][0][1], simple_add)
test_eq(p['native_tools'], ())
test_eq(claude_mk_payload([], 'm', web_search_options='l')['native_tools'], SERVER_TOOLS)
p['tools'][0][0]

A dict-shaped proxy namespace that is empty as a dict still pairs its tools:

In [ ]:
class _ProxyNS(dict):
    def get(self, k, default=None): return simple_add
test_eq(len(claude_mk_payload([], 'm', tools=[lite_mk_func(simple_add)], ns=_ProxyNS())['tools']), 1)


The stream adapter drives one `ClaudeRun` and merges two sources into FastLLM's stream: partial events, re-indexed onto one global block sequence and normalized to `Delta`s (every tool call marked server-executed, names unqualified); and real tool results, lifted from the run's full `user` events, their text passed through `unwrap_typed` so a `str` subclass such as `FullResponse` comes back typed. The collector's placeholder result for an in-process tool is suppressed in favor of the real one, and the final `Completion`'s message gets each result spliced in after its call, so history holds the whole loop:

In [ ]:
#| export
def _reidx():
    "Stateful rebase of per-message block indices onto one global sequence"
    base,mx = 0,-1
    def f(ev):
        nonlocal base,mx
        t = ev.get('type')
        if t=='message_start': base,mx = base+mx+1,-1
        elif t in ('content_block_start','content_block_delta','content_block_stop') and 'index' in ev:
            mx = max(mx, ev['index'])
            ev = {**ev, 'index': ev['index']+base}
        return ev
    return f

def _unq(nm): return nm[len(MCP_PREFIX):] if nm and nm.startswith(MCP_PREFIX) else nm

def _tr_part(b, names):
    "A real `ToolResult` part for one `tool_result` block from the run's user events"
    c = b.get('content','')
    txt = c if isinstance(c, str) else '\n'.join(x.get('text','') for x in c if x.get('type')=='text')
    tid = b.get('tool_use_id')
    return ToolResult(id=tid, name=names.get(tid), text=unwrap_typed(txt), server=True)

The unwrap is what keeps a no-truncation tool result lossless across turns. A tool that returns `FullResponse` does so because every byte matters (a hash-addressed file view, say). `tool_content` marks it on the wire, `_tr_part` restores the real class, and the serialized history keeps the whole text where a plain string would be cut at the cap:

In [ ]:
full = FullResponse('x '*2000)
p = _tr_part(dict(tool_use_id='t1', content=wrap_typed(full)), {'t1': 'lnhashview_file'})
test_eq(type(p.text), FullResponse)
h = [Msg('assistant', [ToolUse(id='t1', name='lnhashview_file', arguments={}, server=True)]), Msg('tool', [p])]
assert str(full) in hist2fmt(h)
type(p.text)

In [ ]:
#| export
def _splice(c, results):
    "Insert each real result after its call in a `Completion`'s message content"
    parts = []
    for p in c.message.content:
        parts.append(p)
        if isinstance(p, ToolUse) and p.id in results: parts.append(results[p.id])
    c.message.content = parts
    return c

`_splice` gives history real results where the collector placed only calls. The stream adapter itself is a queue merge: the collector's outputs and the run's real tool results, in arrival order, with an error result raised as a provider `APIError` after the stream completes:

In [ ]:
#| export
async def claude_acollect_stream(payload, **kwargs):
    "Adapt one `ClaudeRun`'s raw events to FastLLM's delta stream; tools run in-process, results stream real"
    run,results,q = astream(**payload),{},asyncio.Queue()
    ours = {s['name'] for s,_ in (payload.get('tools') or [])}
    f = _reidx()
    async def _deltas():
        opens,saw = {},set()
        async for m in run:
            t = m.get('type')
            if t=='stream_event':
                ev = f(m['event'])
                et,idx = ev.get('type'), ev.get('index')
                d = norm_sse_event(ev)
                for tc in (d.tool_calls or []): tc.server,tc.name = True,_unq(tc.name)
                if et=='content_block_start' and d.tool_calls: opens[idx] = d.tool_calls[0]
                elif et=='content_block_delta' and d.tool_calls and (nested_idx(ev, 'delta', 'partial_json') or '').endswith('}'): saw.add(idx)
                yield d
                if et=='content_block_stop' and (tc := opens.pop(idx, None)) is not None and idx not in saw:
                    yield Delta(tool_calls=[ToolUse(id=tc.id, name=tc.name, arguments={}, server=True)], raw=dict(index=idx))
            elif t=='user':
                c = nested_idx(m, 'message', 'content')
                if isinstance(c, list):
                    for b in c:
                        if b.get('type')=='tool_result':
                            results[b['tool_use_id']] = r = _tr_part(b, run._names)
                            q.put_nowait(r)
    async def _collect():
        try:
            async for o in mk_acollect_stream(_deltas(), index_fn=delta_index_fn, api_name='claude_code', **kwargs):
                if isinstance(o, ToolResult) and o.server and o.name in ours and o.id not in results: continue
                q.put_nowait(_splice(o, results) if isinstance(o, Completion) else o)
        finally: q.put_nowait(None)
    t = asyncio.create_task(_collect())
    try:
        while (o := await q.get()) is not None: yield o
        await t
        if run.result and run.result.get('is_error'):
            raise APIError(str(run.result.get('result') or run.result.get('subtype')), provider='claude_code',
                model=payload.get('model'), status_code=run.result.get('api_error_status'), raw=run.result)
    finally:
        t.cancel()
        await run.aclose()

`ClaudeCodeCallback` is the chat-side glue, inert for every other provider. Before each request it injects the chat's tool namespace into the payload kwargs, which is how the callables reach `claude_mk_payload`. After each response it reshapes the turn's single collected message into the assistant/tool message shape the external loop used to produce, with real results and the server marks dropped, so `chat.full()`, dialog serialization, and the next request's history all work unchanged. The completion itself keeps its server-marked calls, which is what stops FastLLM's own loop from running the tools a second time:

In [ ]:
#| export
def _reshape_hist(chat):
    "Split the turn's collected message into assistant/tool messages, external-loop shaped"
    m = chat.hist[-1]
    if not any(isinstance(p, ToolResult) for p in m.content): return
    out,cur = [],[]
    for p in m.content:
        if isinstance(p, ToolResult):
            if cur:
                out.append(Msg('assistant', cur))
                cur = []
            if out and out[-1].role=='tool': out[-1].content.append(p.replace(server=False))
            else: out.append(Msg('tool', [p.replace(server=False)]))
        else: cur.append(p.replace(server=False) if isinstance(p, ToolUse) and p.server else p)
    if cur: out.append(Msg('assistant', cur))
    chat.hist[-1:] = out

class ClaudeCodeCallback(ChatCallback):
    "Inject the tool namespace into claude_code payloads, and reshape each turn's history"
    order = 20
    async def before_acomplete(self):
        if self.api_name=='claude_code' and self.ns is not None: self.chat.turn_kwargs['ns'] = self.ns
        if False: yield
    async def after_acomplete(self):
        if self.api_name=='claude_code': _reshape_hist(self.chat)
        if False: yield

if not any(getattr(cb, '__name__', '')=='ClaudeCodeCallback' for cb in defaults.chat_callbacks):
    defaults.chat_callbacks.append(ClaudeCodeCallback)

In [ ]:
#| export
api_registry.register('claude_code',
    norm_tool_calls=norm_tool_calls, norm_parts=norm_parts, norm_finish=norm_finish, norm_usage=norm_usage,
    finalize_usage=finalize_usage, mk_payload=claude_mk_payload, acollect_stream=claude_acollect_stream, cost=cost)

The reshape is pure, so its contract shows without a model: one collected message with two tool exchanges becomes the familiar five-message shape, calls unmarked and paired with their results, ready for `hist2fmt` and the next request:

In [ ]:
turn = Msg('assistant', [Thinking('…'), ToolUse(id='t1', name='add', server=True),
    ToolResult(id='t1', name='add', text='4', server=True), Text('4. Now doubling.'),
    ToolUse(id='t2', name='add', server=True), ToolResult(id='t2', name='add', text='8', server=True), Text('8.')])
fchat = AttrDict(hist=[turn])
_reshape_hist(fchat)
test_eq([m.role for m in fchat.hist], ['assistant','tool','assistant','tool','assistant'])
test_eq(fchat.hist[0].content[1].server, False)
test_eq(fchat.hist[1].content[0].text, '4')
test_eq(turn.content[1].server, True)
[(m.role, [type(p).__name__ for p in m.content]) for m in fchat.hist]

## Live runs

Against the real CLI (genuine captured outputs; spends tokens, so out of automated runs): a chat with one Python tool streams thinking, the call, the real result, and the answer, all inside one FastLLM step, and the reshaped history serializes with real results in its `{.tool}` blocks:

In [ ]:
#| eval: false
chat = AsyncChat('claude_code/claude-sonnet-4-6', tools=[simple_add])
rs = await chat('What is 7+3? Use the tool, then answer with only the number.', stream=True, max_steps=3)
parts = [o async for o in rs]
test_eq([m.role for m in chat.hist], ['user','assistant','tool','assistant'])
test_eq(chat.hist[-1].text, '10')
test('"result": "10"', chat.full(), in_)
[type(o).__name__ for o in parts]

['Thinking', 'ToolUse', 'ToolResult', 'Text', 'Completion']

Stopping is the consumers' existing gesture, unchanged: cancel the consumer, close the stream. The close reaches `ClaudeRun.aclose`, which interrupts natively; Claude cancels the pending tool call, so the in-flight callable sees `CancelledError` within about a second, and the process is drained and reaped. Nothing upstream needed a new API:

In [ ]:
#| eval: false
seen = []
async def slow_tool() -> str:
    "Measure very slowly."
    try:
        await asyncio.sleep(60)
        return 'done'
    except asyncio.CancelledError:
        seen.append('cancelled')
        raise

In [ ]:
#| eval: false
ichat = AsyncChat('claude_code/claude-sonnet-4-6', tools=[slow_tool])
irs = await ichat('Use the slow_tool now.', stream=True, max_steps=3)
async for o in irs:
    if type(o).__name__=='ToolUse':
        await asyncio.sleep(1)
        break
await irs.aclose()
await asyncio.sleep(1)
test_eq(seen, ['cancelled'])
seen

['cancelled']